<a href="https://colab.research.google.com/github/abduyea/Career-Trends-Analyzer/blob/main/notebooks/data_cleaning_and_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Cleaning and Preprocessing

loads the master job-postings dataset, performs basic data cleaning, and runs exploratory data analysis (EDA) to prepare a clean datasets


In [13]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Career-Trends-Analyzer").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import (
    mount_drive,
    load_postings,
    load_companies,
    load_jobs,
    load_mappings,
    build_master,
)
from src.config import ROOT_DIR, RAW_DIR, DATA_DIR
from src.utils import peek, missing_summary, summarize_tables

import pandas as pd

print("ROOT_DIR:", ROOT_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT_DIR: /content/drive/MyDrive/Career-Trends-Analyzer


Load Data and Build Master

In [14]:
mount_drive()

postings = load_postings()
companies = load_companies()
jobs = load_jobs()
mappings = load_mappings()

master = build_master(postings, companies, jobs, mappings)

print("Raw master shape:", master.shape)
peek(master)

# work on a copy
df = master.copy()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Raw master shape: (123849, 47)
(123849, 47)


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,...,zip_code_company,address,url,salary_id,max_salary_salary,med_salary_salary,min_salary_salary,pay_period_salary,currency_salary,compensation_type_salary
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,...,07302,242 Tenth Street,https://www.linkedin.com/company/corcoran-sawy...,18531.0,20.0,NaN,17.0,HOURLY,USD,BASE_SALARY
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,...,NaN,NaN,NaN,8059.0,50.0,NaN,30.0,HOURLY,USD,BASE_SALARY
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,...,45227,6880 Wooster Pike,https://www.linkedin.com/company/the-national-...,14949.0,65000.0,NaN,45000.0,YEARLY,USD,BASE_SALARY
3,23221523,"Abrams Fensterman, LLP",Senior Elder Law / Trusts and Estates Associat...,Senior Associate Attorney - Elder Law / Trusts...,175000.0,YEARLY,"New Hyde Park, NY",766262.0,16.0,NaN,...,11042,3 Dakota Drive,https://www.linkedin.com/company/abrams-fenste...,11204.0,175000.0,NaN,140000.0,YEARLY,USD,BASE_SALARY
4,35982263,NaN,Service Technician,Looking for HVAC service tech with experience ...,80000.0,YEARLY,"Burlington, IA",NaN,3.0,NaN,...,NaN,NaN,NaN,20809.0,80000.0,NaN,60000.0,YEARLY,USD,BASE_SALARY


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123849 entries, 0 to 123848
Data columns (total 47 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   job_id                      123849 non-null  int64  
 1   company_name                122130 non-null  object 
 2   title                       123849 non-null  object 
 3   description                 123842 non-null  object 
 4   max_salary                  29793 non-null   float64
 5   pay_period                  36073 non-null   object 
 6   location                    123849 non-null  object 
 7   company_id                  122132 non-null  float64
 8   views                       122160 non-null  float64
 9   med_salary                  6280 non-null    float64
 10  min_salary                  29793 non-null   float64
 11  formatted_work_type         123849 non-null  object 
 12  applies                     23320 non-null   float64
 13  original_liste

In [16]:
# Summary for numeric columns
df.describe(include='all').T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
job_id,123849.0,NaN,NaN,NaN,3896402138.074615,84043545.161881,921716.0,3894586595.0,3901998406.0,3904707077.0,3906267224.0
company_name,122130,24428,Liberty Healthcare and Rehabilitation Services,1108,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,123849,72521,Sales Manager,673,NaN,NaN,NaN,NaN,NaN,NaN,NaN
description,123842,107827,Position Summary: Our Sales Manager has managi...,474,NaN,NaN,NaN,NaN,NaN,NaN,NaN
max_salary,29793.0,NaN,NaN,NaN,91939.423461,701110.138622,1.0,48.28,80000.0,140000.0,120000000.0
pay_period,36073,5,YEARLY,20628,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location,123849,8526,United States,8125,NaN,NaN,NaN,NaN,NaN,NaN,NaN
company_id,122132.0,NaN,NaN,NaN,12204012.335015,25541431.65742,1009.0,14352.0,226965.0,8047188.0,103472979.0
views,122160.0,NaN,NaN,NaN,14.618247,85.903598,1.0,3.0,4.0,8.0,9975.0
med_salary,6280.0,NaN,NaN,NaN,22015.619876,52255.873846,0.0,18.94,25.5,2510.5,750000.0


In [18]:

def column_summary(df: pd.DataFrame) -> pd.DataFrame:
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "missing_count": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2),
        "unique_values": df.nunique(),
        "sample_values": df.apply(lambda x: x.dropna().unique()[:5])
    })
    return summary

column_summary(df)


,dtype,missing_count,missing_percent,unique_values,sample_values
job_id,int64,0,0.00,123849,"[921716, 1829192, 10998357, 23221523, 35982263]"
company_name,object,1719,1.39,24428,"[Corcoran Sawyer Smith, The National Exemplar ..."
title,object,0,0.00,72521,"[Marketing Coordinator, Mental Health Therapis..."
description,object,7,0.01,107827,[Job descriptionA leading real estate firm in ...
max_salary,float64,94056,75.94,5321,"[20.0, 50.0, 65000.0, 175000.0, 80000.0]"
pay_period,object,87776,70.87,5,"[HOURLY, YEARLY, MONTHLY, WEEKLY, BIWEEKLY]"
location,object,0,0.00,8526,"[Princeton, NJ, Fort Collins, CO, Cincinnati, ..."
company_id,float64,1717,1.39,24474,"[2774458.0, 64896719.0, 766262.0, 1481176.0, 8..."
views,float64,1689,1.36,684,"[20.0, 1.0, 8.0, 16.0, 3.0]"
med_salary,float64,117569,94.93,1417,"[350.0, 25.0, 23.0, 56.41, 4200.0]"


In [19]:
# Missing values count
df.isnull().sum()

# Missing value percentage
(df.isnull().mean() * 100).round(2)


,0
job_id,0.00
company_name,1.39
title,0.00
description,0.01
max_salary,75.94
pay_period,70.87
location,0.00
company_id,1.39
views,1.36
med_salary,94.93
